In [1]:


import anndata

from scMEDAL.utils.splitter import DataSplitter
from scMEDAL.utils.utils import read_adata
from scMEDAL.utils.model_train_utils import load_data

import glob

from config_split_paths import data_path,data2split_foldername,folder_splits

import os

2025-01-23 13:34:47.421873: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcudart.so.10.1


data_base_path: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data
outputs_path: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../outputs/AML_outputs


In [2]:
# read paths
adata2split_path = data_path +"/"+folder_splits
print(adata2split_path)
splits_list = glob.glob(adata2split_path+"/*")
glob.glob(splits_list[0]+"/train/*")

/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes/splits


['/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes/splits/split_1/train/exprMatrix.npy',
 '/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes/splits/split_1/train/meta.csv',
 '/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes/splits/split_1/train/geneids.csv']

In [3]:
# get original donor data
adata2split_path_original = data_path + "/" + data2split_foldername
# X,var,obs = read_adata(adata2split_path_original)
X,var,obs = read_adata(adata2split_path_original, issparse=False)
adata = anndata.AnnData(X,obs=obs,var=var)

Reading data from: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes


/project/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_ARMED_2/lib/python3.8/site-packages/anndata/_core/anndata.py:119: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [4]:
# Creating a dict with split paths and adata_splits
def get_splits_dict(splits_path,issparse=False):
    splits_list = glob.glob(splits_path+"/*")
    #print(splits_list)
    split_paths_dict = {}
    adata_splits_dict = {}
    for split in splits_list:
        split_name = split.split("/")[-1]
        split_paths_dict[split_name]={"train":split+"/train","val":split+"/val","test":split+"/test"}
        print(split_paths_dict[split_name])
        adata_splits_dict[split_name]=load_data(split_paths_dict[split_name], eval_test=True, scaling=None,issparse=issparse)
    return split_paths_dict,adata_splits_dict

In [5]:
def check_fold_leakage(adata_splits_dict):
    val_indices = []
    test_indices = []

    # Extracting indices for validation and test sets from each fold in the dictionary
    for split_key in adata_splits_dict.keys():
        fold_data = adata_splits_dict[split_key]
        val_indices.append(set(fold_data['val'].obs['original_index']))
        test_indices.append(set(fold_data['test'].obs['original_index']))

    # Checking for any overlaps (leakage) in validation and test sets
    num_folds = len(adata_splits_dict)
    overlap_detected = False
    for i in range(num_folds):
        for j in range(i + 1, num_folds):
            val_intersection = val_indices[i].intersection(val_indices[j])
            test_intersection = test_indices[i].intersection(test_indices[j])
            if val_intersection:
                print(f"Overlap detected in validation sets between folds {i} and {j}: {len(val_intersection)} common elements")
                overlap_detected = True
            if test_intersection:
                print(f"Overlap detected in test sets between folds {i} and {j}: {len(test_intersection)} common elements")
                overlap_detected = True

    if overlap_detected:
        return False
    else:
        print("No overlap detected in validation and test sets between folds.")
        return True


In [6]:
# Creating a dict with split paths and adata_splits
split_paths_dict,adata_splits_dict = get_splits_dict(data_path+"/"+folder_splits,issparse=False) 

{'train': '/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes/splits/split_1/train', 'val': '/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes/splits/split_1/val', 'test': '/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes/splits/split_1/test'}
Reading data from: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes/splits/split_1/train
X.shape before scaling (23049, 2916)
Reading data from: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_data/log_transformed_2916hvggenes/splits/split_1/val
X.shape before scaling (7684, 2916)
Reading data from: /

In [7]:
data_path+folder_splits

'/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/AML/../data/AML_datalog_transformed_2916hvggenes/splits'

In [8]:


# Check if there is any overlap between 
overlap_check_result = check_fold_leakage(adata_splits_dict)

No overlap detected in validation and test sets between folds.


In [9]:
# get train data from split 1
split_to_check = "split_5"
adata_train = adata_splits_dict[split_to_check]["train"]
print("adata_train shape",adata_train.shape)
# read test data from split 1
adata_test = adata_splits_dict[split_to_check]["test"]
print("adata_test shape",adata_test.shape)
# read val data from split 1
adata_val = adata_splits_dict[split_to_check]["val"]
print("adata_val shape",adata_val.shape)

adata_train shape (23050, 2916)
adata_test shape (7683, 2916)
adata_val shape (7684, 2916)


In [10]:
adata_test.obs

,Unnamed: 0,Unnamed: 0.1,Unnamed: 0.1.1,Cell,NumberOfReads,AlignedToGenome,AlignedToTranscriptome,TranscriptomeUMIs,NumberOfGenes,CyclingScore,...,Score_NK,NanoporeTranscripts,id,Day,unique_id,Patient_group,celltype,batch,n_genes,original_index
0,9561,10273,10273,AML707B-D18_CCTACCAGGCAT,64368,49649,26593,2312,855,-0.431000,...,0.095,NaN,AML707B,D18,AML707B_D18,AML,T,AML707B,855,9561
1,32476,33537,33537,AML420B-D0_TGTACGCCAGAN,36626,18140,9603,1155,526,-0.287000,...,0.073,NaN,AML420B,D0,AML420B_D0,AML,T,AML420B,526,32476
2,26410,27471,27471,AML328-D29_CTTAAACTAGTC,116885,74271,35684,10829,3349,1.948000,...,0.020,NaN,AML328,D29,AML328_D29,AML,Prog-like,AML328,3349,26410
3,24349,25223,25223,OCI-AML3_ACGTTTGGCAAG,18260,14490,11423,2685,1419,0.490586,...,0.012,NaN,OCI,NaN,OCI,cellline,GMP-like,OCI,1419,24349
4,28335,29396,29396,AML921A-D0_AACTACTCAGTC,32773,23824,12237,2285,706,-0.404000,...,0.021,NaN,AML921A,D0,AML921A_D0,AML,GMP-like,AML921A,706,28335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7678,18431,19305,19305,AML329-D0_CCCGAGTCAGAT,58129,43251,26296,1652,650,-0.331000,...,0.077,NaN,AML329,D0,AML329_D0,AML,T,AML329,650,18431
7679,18942,19816,19816,AML556-D31_ACGGTTTTTTAA,47503,41529,28370,3787,1454,-0.412000,...,0.007,NaN,AML556,D31,AML556_D31,AML,earlyEry,AML556,1454,18942
7680,27480,28541,28541,AML328-D29_AGGCGACTTACG,15810,9456,6562,1512,705,-0.478000,...,0.113,NaN,AML328,D29,AML328_D29,AML,T,AML328,705,27480
7681,19769,20643,20643,AML556-D31_TTGACAGGGTGA,47889,41266,20026,2340,987,-0.530000,...,0.012,NaN,AML556,D31,AML556_D31,AML,Mono,AML556,987,19769


In [11]:
# check that train, test and val should sum up total data
adata_train.shape[0]+adata_val.shape[0]+adata_test.shape[0]

38417

In [12]:
import numpy as np
# # check all donors list
adata.obs['batch'] = adata.obs["batch"].astype('category')
donor_list = list(np.unique(adata.obs["batch"]))
# Seen donor list
donor_seen_list = list(np.unique(adata_train.obs["batch"]))
print(donor_seen_list)

['AML1012', 'AML210A', 'AML328', 'AML329', 'AML419A', 'AML420B', 'AML475', 'AML556', 'AML707B', 'AML870', 'AML916', 'AML921A', 'BM1', 'BM2', 'BM3', 'BM4', 'BM5', 'MUTZ3', 'OCI']


In [13]:
# check if the stratification worked

# call the splitter stratification test
splitter = DataSplitter()
comp_df = splitter.check_stratification(adata, adata_train, adata_val, adata_test, ['batch', 'celltype'])
# add donor and celltype columns
comp_df["batch"] = [x.split("_")[0] for x in comp_df.index ]
comp_df["celltype"] = [x.split("_")[1] for x in comp_df.index ]


# Donor is in seen list
condition1 = comp_df["batch"].isin(donor_seen_list)
# Values equal to zero
condition2 = comp_df[['Train', 'Validation', 'Test']].eq(0).any(axis=1)
filtered_df = comp_df.loc[condition1 & condition2]
# We want to see the seen donors well distributed trough the train, test and validation datasets. No zero entries.


In [14]:
filtered_df

,Original,Train,Validation,Test,batch,celltype
AML1012_NK,0.000078,0.000087,0.00013,0.00000,AML1012,NK
AML1012_Plasma,0.000026,0.000000,0.00000,0.00013,AML1012,Plasma
AML1012_ProB,0.000052,0.000087,0.00000,0.00000,AML1012,ProB
AML210A_HSC,0.000078,0.000087,0.00000,0.00013,AML210A,HSC
AML210A_ProB,0.000026,0.000043,0.00000,0.00000,AML210A,ProB
AML328_GMP,0.000078,0.000130,0.00000,0.00000,AML328,GMP
AML419A_GMP,0.000026,0.000000,0.00000,0.00013,AML419A,GMP
AML419A_HSC,0.000052,0.000043,0.00000,0.00013,AML419A,HSC
AML419A_NK,0.000052,0.000000,0.00013,0.00013,AML419A,NK
AML419A_Plasma,0.000052,0.000087,0.00000,0.00000,AML419A,Plasma


In [15]:
adata_val

AnnData object with n_obs × n_vars = 7684 × 2916
    obs: 'Unnamed: 0', 'Unnamed: 0.1', 'Unnamed: 0.1.1', 'Cell', 'NumberOfReads', 'AlignedToGenome', 'AlignedToTranscriptome', 'TranscriptomeUMIs', 'NumberOfGenes', 'CyclingScore', 'CyclingBinary', 'MutTranscripts', 'WtTranscripts', 'PredictionRF2', 'PredictionRefined', 'CellType', 'Score_HSC', 'Score_Prog', 'Score_GMP', 'Score_ProMono', 'Score_Mono', 'Score_cDC', 'Score_pDC', 'Score_earlyEry', 'Score_lateEry', 'Score_ProB', 'Score_B', 'Score_Plasma', 'Score_T', 'Score_CTL', 'Score_NK', 'NanoporeTranscripts', 'id', 'Day', 'unique_id', 'Patient_group', 'celltype', 'batch', 'n_genes', 'original_index', 'combined_stratify_col'
    var: 'Unnamed: 0'

In [16]:
adata_train.obs["combined_stratify_col"]

0                 AML328_T
1               AML329_cDC
2          AML329_GMP-like
3             AML707B_Mono
4                AML707B_T
               ...        
23045    AML921A_Prog-like
23046             AML328_T
23047             AML328_T
23048     AML707B_GMP-like
23049             AML328_T
Name: combined_stratify_col, Length: 23050, dtype: object

In [17]:
adata_train.obs["combined_stratify_col"]

0                 AML328_T
1               AML329_cDC
2          AML329_GMP-like
3             AML707B_Mono
4                AML707B_T
               ...        
23045    AML921A_Prog-like
23046             AML328_T
23047             AML328_T
23048     AML707B_GMP-like
23049             AML328_T
Name: combined_stratify_col, Length: 23050, dtype: object

In [18]:
# from matplotlib import pyplot as plt
# index_val = adata_val.obs.loc[adata_val.obs['combined_stratify_col']=='5163_Neu-mat'].index
# print("index_val.shape",index_val.shape)
# plt.hist(adata_val.X[np.array(index_val.astype(int)),:])


In [19]:
adata_train.obs

,Unnamed: 0,Unnamed: 0.1,Unnamed: 0.1.1,Cell,NumberOfReads,AlignedToGenome,AlignedToTranscriptome,TranscriptomeUMIs,NumberOfGenes,CyclingScore,...,NanoporeTranscripts,id,Day,unique_id,Patient_group,celltype,batch,n_genes,original_index,combined_stratify_col
0,6014,6726,6726,AML328-D113_AGCCGAGGGACC,25577,18744,13048,1251,534,-0.343,...,NaN,AML328,D113,AML328_D113,AML,T,AML328,534,6014,AML328_T
1,15392,16266,16266,AML329-D20_TTTTTGGACCAN,136195,121726,80027,3625,1543,-0.505,...,NaN,AML329,D20,AML329_D20,AML,cDC,AML329,1543,15392,AML329_cDC
2,18500,19374,19374,AML329-D0_GCAGATCCCGGC,79117,55642,32700,2723,995,-0.575,...,NaN,AML329,D0,AML329_D0,AML,GMP-like,AML329,995,18500,AML329_GMP-like
3,4471,5183,5183,AML707B-D41_CGCACGAGGAAC,44397,30071,15709,1817,890,-0.491,...,NaN,AML707B,D41,AML707B_D41,AML,Mono,AML707B,890,4471,AML707B_Mono
4,10376,11088,11088,AML707B-D18_TGGTCCATAATG,36284,28360,18354,1846,948,-0.577,...,NaN,AML707B,D18,AML707B_D18,AML,T,AML707B,948,10376,AML707B_T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23045,28693,29754,29754,AML921A-D0_ACTTTTCGCACT,30883,21822,15711,2322,1023,-0.545,...,NaN,AML921A,D0,AML921A_D0,AML,Prog-like,AML921A,1023,28693,AML921A_Prog-like
23046,37819,40492,40492,AML328-D171_TATACATTGAGC,11418,8337,4753,1056,504,-0.309,...,NaN,AML328,D171,AML328_D171,AML,T,AML328,504,37819,AML328_T
23047,37194,39867,39867,AML328-D171_ATCTCGCCTAGG,16389,12160,6647,1367,660,-0.390,...,NaN,AML328,D171,AML328_D171,AML,T,AML328,660,37194,AML328_T
23048,16850,17724,17724,AML707B-D0_ACGCTATGTCTC,45613,33517,22856,3691,1510,-0.471,...,NaN,AML707B,D0,AML707B_D0,AML,GMP-like,AML707B,1510,16850,AML707B_GMP-like
